# Exploratory Data Analysis: E-commerce Customer Lifetime Value Prediction

This notebook repairs the prior malformed notebook structure and preserves the intended EDA flow for customer lifetime value (CLV) modeling. It reviews data quality, target behavior, feature relationships, nonlinearity, segmentation, and an explicit mapping from EDA findings to modeling actions.


In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy import stats
from sklearn.feature_selection import mutual_info_regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import PolynomialFeatures

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

EXPECTED_DATA_FILE = 'synthetic_data_86 (ecommerce data set).csv'
TARGET_COL = 'estimated_lifetime_value'


def resolve_data_path(expected_name: str) -> Path:
    expected_path = Path(expected_name)
    if expected_path.exists():
        return expected_path

    csv_candidates = sorted(Path('.').glob('*.csv'))
    if len(csv_candidates) == 1:
        print(f"Expected '{expected_name}' not found. Falling back to '{csv_candidates[0]}'.")
        return csv_candidates[0]

    raise FileNotFoundError(
        f"Could not find '{expected_name}'. Place the dataset in the notebook directory "
        "or provide exactly one CSV file for automatic fallback."
    )


data_path = resolve_data_path(EXPECTED_DATA_FILE)
df = pd.read_csv(data_path)
continuous_features = df.select_dtypes(include=[np.number]).columns.drop(TARGET_COL).tolist()
categorical_features = df.select_dtypes(exclude=[np.number]).columns.tolist()

target = df[TARGET_COL]
y = target.to_numpy()

print(f"Loaded dataset: {data_path}")
print(f"Dataset shape: {df.shape}")
print(f"Continuous features: {continuous_features}")
print(f"Categorical features: {categorical_features}")
display(df.head())


## 1. Dataset Overview and Data Quality

Start by confirming the dataset schema, completeness, and feature definitions before interpreting any modeling signals.


In [ ]:
feature_descriptions = pd.DataFrame(
    {
        'feature': [
            'total_purchase_count',
            'average_order_value',
            'days_since_first_purchase',
            'days_since_last_purchase',
            'product_category_diversity',
            'loyalty_program_membership',
            TARGET_COL,
        ],
        'description': [
            'Total number of purchases made by the customer',
            "Mean value of a customer's orders",
            'Customer tenure in days',
            'Recency in days since the latest purchase',
            'Normalized breadth of product category engagement',
            'Observed loyalty enrollment status',
            'Estimated customer lifetime value (target)',
        ],
    }
)

missing_summary = pd.DataFrame(
    {
        'missing_values': df.isna().sum(),
        'dtype': df.dtypes.astype(str),
        'unique_values': df.nunique(),
    }
)

print('Dataset info:')
df.info()
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"Missing values across all columns: {int(df.isna().sum().sum())}")

display(feature_descriptions)
display(missing_summary)
display(df.describe(include='all').transpose())


## 2. Target Variable Analysis

Because CLV is typically right-skewed, the target section pairs visual checks with measurable diagnostics so any transformation recommendation is easy to audit.


In [ ]:
quartiles = target.quantile([0.25, 0.5, 0.75])
Q1, median_target, Q3 = quartiles.loc[0.25], quartiles.loc[0.5], quartiles.loc[0.75]
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outliers = target[(target < lower_bound) | (target > upper_bound)]
outliers_zscore = int((np.abs(stats.zscore(target)) > 3).sum())

target_overview = pd.DataFrame(
    {
        'metric': ['mean', 'median', 'std', 'min', 'max', 'range', 'skewness', 'kurtosis', 'Q1', 'Q3', 'IQR'],
        'value': [
            target.mean(),
            target.median(),
            target.std(),
            target.min(),
            target.max(),
            target.max() - target.min(),
            stats.skew(target),
            stats.kurtosis(target),
            Q1,
            Q3,
            IQR,
        ],
    }
)

display(target_overview.round(4))
print(f"IQR outliers: {len(outliers)} observations ({100 * len(outliers) / len(target):.1f}%)")
print(f"Z-score outliers (> 3): {outliers_zscore} observations ({100 * outliers_zscore / len(target):.1f}%)")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].hist(target, bins=40, color='steelblue', edgecolor='black', alpha=0.75)
axes[0, 0].axvline(target.mean(), color='red', linestyle='--', linewidth=2, label=f"Mean: {target.mean():.2f}")
axes[0, 0].axvline(target.median(), color='green', linestyle='--', linewidth=2, label=f"Median: {target.median():.2f}")
axes[0, 0].set_title('Raw CLV Distribution')
axes[0, 0].set_xlabel('Estimated Lifetime Value')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].legend()

log_target = np.log1p(target)
axes[0, 1].hist(log_target, bins=40, color='coral', edgecolor='black', alpha=0.75)
axes[0, 1].set_title('Log1p CLV Distribution')
axes[0, 1].set_xlabel('log1p(Estimated Lifetime Value)')
axes[0, 1].set_ylabel('Frequency')

stats.probplot(target, dist='norm', plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot: Raw CLV')

axes[1, 1].boxplot(target, vert=True)
axes[1, 1].set_title('CLV Box Plot')
axes[1, 1].set_ylabel('Estimated Lifetime Value')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('target_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved figure: target_distribution.png")


In [ ]:
def summarize_distribution(series: pd.Series) -> pd.Series:
    q1 = series.quantile(0.25)
    median = series.quantile(0.50)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    return pd.Series(
        {
            'mean': series.mean(),
            'median': median,
            'std': series.std(),
            'skewness': stats.skew(series),
            'kurtosis': stats.kurtosis(series),
            'IQR': iqr,
            'IQR/median': np.nan if median == 0 else iqr / median,
        }
    )

transformation_diagnostics = pd.DataFrame(
    {
        'raw_target': summarize_distribution(target),
        'log1p_target': summarize_distribution(log_target),
    }
).T

transformation_diagnostics['abs_skewness'] = transformation_diagnostics['skewness'].abs()
transformation_diagnostics['abs_kurtosis'] = transformation_diagnostics['kurtosis'].abs()

display(transformation_diagnostics.round(4))

skew_improvement = transformation_diagnostics.loc['raw_target', 'abs_skewness'] - transformation_diagnostics.loc['log1p_target', 'abs_skewness']
kurtosis_improvement = transformation_diagnostics.loc['raw_target', 'abs_kurtosis'] - transformation_diagnostics.loc['log1p_target', 'abs_kurtosis']
iqr_ratio_change = transformation_diagnostics.loc['raw_target', 'IQR/median'] - transformation_diagnostics.loc['log1p_target', 'IQR/median']

print('Transformation interpretation:')
if skew_improvement > 0 and kurtosis_improvement > 0:
    print(
        f"- log1p materially improves shape: |skew| drops by {skew_improvement:.2f} and |kurtosis| drops by {kurtosis_improvement:.2f}."
    )
else:
    print('- log1p does not uniformly improve both skewness and kurtosis, so any transformation choice should be validated with model metrics.')

if iqr_ratio_change > 0:
    print(f"- Dispersion relative to the median also tightens (IQR/median improves by {iqr_ratio_change:.2f}), supporting a more stable modeling scale.")
else:
    print(f"- IQR/median changes by {iqr_ratio_change:.2f}, so the visual improvement should be weighed against predictive performance.")


## 3. Feature Analysis

This section summarizes feature distributions before comparing them to the target.


In [ ]:
continuous_summary = df[continuous_features].describe().T
continuous_summary['skewness'] = df[continuous_features].apply(stats.skew)
continuous_summary['kurtosis'] = df[continuous_features].apply(stats.kurtosis)

display(continuous_summary.round(4))

if categorical_features:
    for feature in categorical_features:
        print(f"\n{feature} distribution:")
        display(pd.DataFrame({'count': df[feature].value_counts(), 'share': df[feature].value_counts(normalize=True).round(4)}))


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(15, 12))
axes = axes.flatten()

for idx, feature in enumerate(continuous_features):
    axes[idx].hist(df[feature], bins=35, color='steelblue', edgecolor='black', alpha=0.75)
    axes[idx].axvline(df[feature].mean(), color='red', linestyle='--', linewidth=1.5, label='Mean')
    axes[idx].axvline(df[feature].median(), color='green', linestyle='--', linewidth=1.5, label='Median')
    axes[idx].set_title(f'Distribution of {feature}')
    axes[idx].set_xlabel(feature)
    axes[idx].set_ylabel('Frequency')
    axes[idx].legend()

for idx in range(len(continuous_features), len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig('continuous_features_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved figure: continuous_features_distributions.png")


In [ ]:
if categorical_features:
    fig, axes = plt.subplots(1, len(categorical_features), figsize=(6 * len(categorical_features), 5))
    if len(categorical_features) == 1:
        axes = [axes]

    for idx, feature in enumerate(categorical_features):
        value_counts = df[feature].value_counts()
        axes[idx].bar(value_counts.index.astype(str), value_counts.values, color=['#1f77b4', '#ff7f0e'][:len(value_counts)], edgecolor='black', alpha=0.75)
        axes[idx].set_title(f'Distribution of {feature}')
        axes[idx].set_xlabel(feature)
        axes[idx].set_ylabel('Count')
        axes[idx].grid(True, axis='y', alpha=0.3)
        for i, value in enumerate(value_counts.values):
            axes[idx].text(i, value + max(value_counts.values) * 0.02, str(value), ha='center')

    plt.tight_layout()
    plt.savefig('categorical_features_distributions.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("Saved figure: categorical_features_distributions.png")


## 4. Relationship Analysis: Features vs Target

Pearson correlation remains a useful baseline, but continuous predictors are also evaluated with Spearman correlation and mutual information to surface monotonic or nonlinear signal that a strictly linear measure can miss.


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(15, 12))
axes = axes.flatten()

for idx, feature in enumerate(continuous_features):
    pearson_r = df[feature].corr(target)
    axes[idx].scatter(df[feature], target, alpha=0.55, s=30, color='steelblue')
    axes[idx].set_title(f'{feature} vs {TARGET_COL}')
    axes[idx].set_xlabel(feature)
    axes[idx].set_ylabel(TARGET_COL)
    axes[idx].grid(True, alpha=0.3)
    axes[idx].text(
        0.03,
        0.96,
        f"Pearson r = {pearson_r:.3f}",
        transform=axes[idx].transAxes,
        va='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.7),
    )

for idx in range(len(continuous_features), len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig('continuous_features_vs_target.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved figure: continuous_features_vs_target.png")


In [ ]:
pearson_scores = df[continuous_features].corrwith(target)
spearman_scores = df[continuous_features].corrwith(target, method='spearman')
mi_scores = mutual_info_regression(df[continuous_features], target, random_state=42)

association_df = pd.DataFrame(
    {
        'feature': continuous_features,
        'pearson_r': pearson_scores.values,
        'spearman_rho': spearman_scores.values,
        'mutual_information': mi_scores,
    }
)
association_df['abs_pearson_r'] = association_df['pearson_r'].abs()
association_df['abs_spearman_rho'] = association_df['spearman_rho'].abs()
association_df['spearman_minus_pearson'] = association_df['abs_spearman_rho'] - association_df['abs_pearson_r']
association_df['pearson_rank'] = association_df['abs_pearson_r'].rank(ascending=False, method='dense').astype(int)
association_df['spearman_rank'] = association_df['abs_spearman_rho'].rank(ascending=False, method='dense').astype(int)
association_df['mi_rank'] = association_df['mutual_information'].rank(ascending=False, method='dense').astype(int)
association_df = association_df.sort_values(['mutual_information', 'abs_spearman_rho', 'abs_pearson_r'], ascending=False).reset_index(drop=True)

display(
    association_df[
        [
            'feature',
            'pearson_r',
            'spearman_rho',
            'mutual_information',
            'pearson_rank',
            'spearman_rank',
            'mi_rank',
            'spearman_minus_pearson',
        ]
    ].round(4)
)

corr_matrix = df[continuous_features + [TARGET_COL]].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.3f', square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Pearson Correlation Matrix: Features and Target', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print('Saved figure: correlation_heatmap.png')

stronger_nonlinear_signal = association_df[
    (association_df['spearman_minus_pearson'] > 0.05)
    | ((association_df['mi_rank'] <= 2) & (association_df['pearson_rank'] > association_df['mi_rank']))
]

print('\nAssociation interpretation:')
if stronger_nonlinear_signal.empty:
    print('- Pearson, Spearman, and mutual information tell a broadly similar story, so the strongest drivers appear mostly monotonic and linear enough for baseline models.')
else:
    for row in stronger_nonlinear_signal.itertuples(index=False):
        print(
            f"- {row.feature}: Spearman exceeds Pearson by {row.spearman_minus_pearson:.3f} and MI={row.mutual_information:.3f}, suggesting monotonic or nonlinear structure stronger than a purely linear fit captures."
        )


In [ ]:
if categorical_features:
    fig, axes = plt.subplots(1, len(categorical_features), figsize=(6 * len(categorical_features), 5))
    if len(categorical_features) == 1:
        axes = [axes]

    for idx, feature in enumerate(categorical_features):
        stats_by_category = df.groupby(feature)[TARGET_COL].agg(['count', 'mean', 'median', 'std']).round(2)
        print(f"\nTarget statistics by {feature}:")
        display(stats_by_category)

        df.boxplot(column=TARGET_COL, by=feature, ax=axes[idx])
        axes[idx].set_title(f'{feature} vs {TARGET_COL}')
        axes[idx].set_xlabel(feature)
        axes[idx].set_ylabel(TARGET_COL)

    plt.suptitle('')
    plt.tight_layout()
    plt.savefig('categorical_features_vs_target.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("Saved figure: categorical_features_vs_target.png")


## 5. Interaction and Nonlinearity Analysis

Interactions, polynomial fit comparisons, and binned trend plots provide additional evidence about whether flexible models or explicit interaction terms are likely to help.


In [ ]:
interactions = []

for i, feat1 in enumerate(continuous_features):
    for feat2 in continuous_features[i + 1:]:
        interaction_term = df[feat1] * df[feat2]
        corr_with_target = interaction_term.corr(target)
        interactions.append(
            {
                'interaction': f'{feat1} × {feat2}',
                'signed_correlation': corr_with_target,
                'abs_correlation': abs(corr_with_target),
            }
        )

interactions_df = pd.DataFrame(interactions).sort_values('abs_correlation', ascending=False).reset_index(drop=True)
print('Top interaction signals:')
display(interactions_df.head(10).round(4))

X_continuous = df[continuous_features].to_numpy()
global_linear_model = LinearRegression().fit(X_continuous, y)
r2_linear = r2_score(y, global_linear_model.predict(X_continuous))

poly_features = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly_features.fit_transform(X_continuous)
global_poly_model = LinearRegression().fit(X_poly, y)
r2_poly = r2_score(y, global_poly_model.predict(X_poly))

curve_evidence = []
for feature in continuous_features:
    x_feature = df[[feature]].to_numpy()
    linear_model = LinearRegression().fit(x_feature, y)
    r2_feature_linear = r2_score(y, linear_model.predict(x_feature))

    feature_poly = PolynomialFeatures(degree=2, include_bias=False)
    x_feature_poly = feature_poly.fit_transform(x_feature)
    poly_model = LinearRegression().fit(x_feature_poly, y)
    r2_feature_poly = r2_score(y, poly_model.predict(x_feature_poly))

    curve_evidence.append(
        {
            'feature': feature,
            'linear_r2': r2_feature_linear,
            'polynomial_r2': r2_feature_poly,
            'r2_gain': r2_feature_poly - r2_feature_linear,
        }
    )

curve_evidence_df = pd.DataFrame(curve_evidence).sort_values('r2_gain', ascending=False).reset_index(drop=True)

print(f"\nGlobal linear R²: {r2_linear:.4f}")
print(f"Global polynomial R²: {r2_poly:.4f}")
print(f"Relative improvement: {(r2_poly - r2_linear) / r2_linear:.2%}")
print('\nUnivariate curve evidence:')
display(curve_evidence_df.round(4))


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(15, 12))
axes = axes.flatten()

for idx, feature in enumerate(continuous_features):
    binned = df[[feature, TARGET_COL]].copy()
    binned['bin'] = pd.qcut(binned[feature], q=10, duplicates='drop')
    binned_stats = binned.groupby('bin').agg(feature_mean=(feature, 'mean'), target_mean=(TARGET_COL, 'mean'), count=(TARGET_COL, 'size'))

    axes[idx].scatter(binned_stats['feature_mean'], binned_stats['target_mean'], s=binned_stats['count'] * 4, alpha=0.7, color='steelblue')
    axes[idx].plot(binned_stats['feature_mean'], binned_stats['target_mean'], color='darkred', linestyle='--', linewidth=2)
    axes[idx].set_title(f'Binned trend: {feature}')
    axes[idx].set_xlabel(feature)
    axes[idx].set_ylabel(f'Mean {TARGET_COL}')
    axes[idx].grid(True, alpha=0.3)

for idx in range(len(continuous_features), len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig('partial_dependence_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved figure: partial_dependence_analysis.png")


## 6. Segmentation Analysis

The segmentation review stays explicitly descriptive: differences across loyalty status or value bands highlight associations in observed behavior, not proof that membership or any one tactic caused higher CLV.


In [ ]:
df['clv_segment'] = pd.qcut(df[TARGET_COL], q=4, labels=['Low', 'Medium', 'High', 'VIP'])

def enrolled_share(series: pd.Series) -> float:
    normalized = series.astype(str).str.lower().str.strip()
    return normalized.eq('enrolled').mean()

loyalty_summary = df.groupby('loyalty_program_membership')[TARGET_COL].agg(['count', 'mean', 'median', 'std', 'min', 'max']).round(2)
segment_summary = (
    df.groupby('clv_segment')
    .agg(
        customer_count=(TARGET_COL, 'size'),
        clv_min=(TARGET_COL, 'min'),
        clv_median=(TARGET_COL, 'median'),
        clv_max=(TARGET_COL, 'max'),
        enrolled_share=('loyalty_program_membership', enrolled_share),
    )
    .round(3)
)
segment_feature_means = df.groupby('clv_segment')[continuous_features].mean().round(2)

print('Observed CLV by loyalty status:')
display(loyalty_summary)

print('Observed CLV segments:')
display(segment_summary)

print('Segment feature means:')
display(segment_feature_means)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for category in df['loyalty_program_membership'].dropna().unique():
    subset = df.loc[df['loyalty_program_membership'] == category, TARGET_COL]
    axes[0].hist(subset, bins=30, alpha=0.6, label=category, edgecolor='black')

axes[0].set_title('CLV Distribution by Loyalty Status')
axes[0].set_xlabel(TARGET_COL)
axes[0].set_ylabel('Frequency')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

loyalty_groups = [df.loc[df['loyalty_program_membership'] == category, TARGET_COL].to_numpy() for category in df['loyalty_program_membership'].dropna().unique()]
axes[1].violinplot(loyalty_groups, positions=np.arange(1, len(loyalty_groups) + 1))
axes[1].set_xticks(np.arange(1, len(loyalty_groups) + 1))
axes[1].set_xticklabels(df['loyalty_program_membership'].dropna().unique())
axes[1].set_title('CLV Distribution by Loyalty Status (Violin)')
axes[1].set_ylabel(TARGET_COL)
axes[1].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('loyalty_segmentation.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved figure: loyalty_segmentation.png")

loyalty_diff = loyalty_summary['mean'].max() - loyalty_summary['mean'].min()
print('\nSegmentation interpretation:')
print(f"- Loyalty segments differ descriptively by about {loyalty_diff:.2f} in average CLV, which is useful for prioritization and targeting discussions.")
print('- This is an observational association, not evidence that loyalty enrollment caused higher CLV.')
print('- Potential confounders include customer tenure, purchase recency, prior value, campaign targeting, and self-selection into the loyalty program.')
print('- Business-facing takeaway: treat loyalty status as a segmentation signal for messaging or retention design, then use experiments or quasi-experimental methods before making causal ROI claims.')


## 7. Traceability: EDA Findings to Modeling Actions

This final section makes each modeling recommendation directly traceable to a specific EDA finding, the evidence supporting it, and the validation step needed to confirm the action adds value.


In [ ]:
top_interaction = interactions_df.iloc[0]
top_curve = curve_evidence_df.iloc[0]
vip_threshold = float(df.loc[df['clv_segment'] == 'VIP', TARGET_COL].min())
segment_counts = df['clv_segment'].value_counts().sort_index().to_dict()

recommendation_map = pd.DataFrame(
    [
        {
            'finding': 'Target is right-skewed with a long upper tail',
            'evidence': f"|skew| {transformation_diagnostics.loc['raw_target', 'abs_skewness']:.2f} → {transformation_diagnostics.loc['log1p_target', 'abs_skewness']:.2f}; IQR/median {transformation_diagnostics.loc['raw_target', 'IQR/median']:.2f} → {transformation_diagnostics.loc['log1p_target', 'IQR/median']:.2f}; {len(outliers)} IQR outliers",
            'modeling_action': 'Model log1p(CLV) and back-transform predictions for reporting',
            'expected_benefit': 'Less tail dominance and more stable residual behavior',
            'validation_check': 'Compare CV RMSE/MAE on the original CLV scale plus residual symmetry',
        },
        {
            'finding': 'Interaction structure is present among purchase behavior variables',
            'evidence': f"Top interaction: {top_interaction['interaction']} (corr={top_interaction['signed_correlation']:.3f})",
            'modeling_action': 'Add the strongest interaction terms or prefer models that learn interactions automatically',
            'expected_benefit': 'Captures compounding effects of frequency, order value, and recency',
            'validation_check': 'Run ablation tests with and without interaction features',
        },
        {
            'finding': 'Nonlinear patterns outperform a linear-only summary',
            'evidence': f"Global R² {r2_linear:.3f} → {r2_poly:.3f}; strongest univariate gain: {top_curve['feature']} (+{top_curve['r2_gain']:.3f})",
            'modeling_action': 'Benchmark tree boosting, splines, or polynomial features against linear baselines',
            'expected_benefit': 'Better fit for curved, threshold-like, and saturation effects',
            'validation_check': 'Compare repeated CV performance and inspect partial dependence / binned trends',
        },
        {
            'finding': 'Value segments show materially different target ranges and loyalty mix',
            'evidence': f"Segment counts {segment_counts}; VIP starts at {vip_threshold:.2f}; loyalty mix changes across segments",
            'modeling_action': 'Use stratified validation on CLV bins/segments to preserve tail representation across folds',
            'expected_benefit': 'More stable evaluation for high-value customers and segment-sensitive errors',
            'validation_check': 'Audit fold-level target quantiles and segment shares before training',
        },
    ]
)

display(recommendation_map)

print('Summary recommendations:')
print('- Use the log1p target as the primary modeling target candidate, but confirm with cross-validated error on the original CLV scale.')
print('- Prioritize recency, frequency, order value, and the strongest interaction terms for feature engineering or flexible learners.')
print('- Treat loyalty segmentation as descriptive business context and validate any intervention ideas with causal designs.')
print('- Keep the saved figures above as the coherent EDA artifact set for the modeling handoff.')


In [ ]:
print('EDA analysis complete.')
print('Generated figure files:')
for filename in [
    'target_distribution.png',
    'continuous_features_distributions.png',
    'categorical_features_distributions.png',
    'continuous_features_vs_target.png',
    'correlation_heatmap.png',
    'categorical_features_vs_target.png',
    'partial_dependence_analysis.png',
    'loyalty_segmentation.png',
]:
    print(f'- {filename}')

print('\nNotebook note: successful execution depends on the CLV CSV being available in the notebook directory or as the only CSV fallback file.')
